In [1]:
import typing as T
from typing import Dict, List, Tuple
import pickle
import json
import os
import pathlib
import pathlib as P
import sys
import pandas as pd
import itertools as it
import functools as ft
import operator as opr
import collections as clt
import math
from math import pi
import re

In [2]:
prj_root = P.Path("__file__").absolute().parent.parent.parent
if (p := str(prj_root)) not in sys.path:
  sys.path.append(p)

In [3]:
import util.metrics as um
import sklearn.metrics as metrics
import notebooks.case_analysis.embedding_analysis as ea

In [4]:
import torch as th
import numpy as np
import scipy as sci
import scipy.stats as stats
import scipy.sparse as sp
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as colors
import scipy.stats as ss
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from tqdm import tqdm
import umap
import openTSNE as S
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches
import goatools as gt
from goatools.base import get_godag
from goatools.gosubdag.gosubdag import GoSubDag
from goatools.gosubdag.plot.gosubdag_plot import GoSubDagPlot

/data0/shaojiangyi/anaconda3/envs/py311_torch240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
ns = ['cc', 'mf', 'bp']
ontology_lst = ["cellular_component",
                "molecular_function",
                "biological_process"]

In [6]:
# load go ic
# load position with count
goic_dir = prj_root / "data" / "data-netgo"
count_path = goic_dir / "term_counts_with_position-111.pkl"
term_count_ic = pd.read_pickle(count_path)

In [7]:
term_count_ic = term_count_ic.astype({
    "gos": str,
    "counts": int,
    "ic": float,
    "position": str,
    "maxdep": int,
    "lgst": int,
})

In [8]:
data_path = [f"/data0/shaojiangyi/pprogo-flg-1/data/data-netgo/{x}_data.pkl"
             for x in ["train", "valid", "test"]]
prot_data = pd.concat([pd.read_pickle(x) for x in data_path], ignore_index=True)

In [9]:
prot_labels = []
for n in ns:
    y_label = prj_root / "data" / n / "label.pkl"
    with open(y_label, "rb") as f:
        labels = pickle.load(f)
    tmp = clt.defaultdict(list)
    for k, v in zip(labels["protein"], labels["go"]):
        tmp[k].append(v)
    prot_labels.append(tmp)

In [10]:
len(prot_labels[2])

91443

In [11]:
label_dir = P.Path("/data0/zhaojianxiang/data/")
curr_labels = []
for n in ns:
    label_path = label_dir / n / "go_list.txt"
    with open(label_path, "r") as f:
        labels = f.read().splitlines()
    curr_labels.append(labels)
namespace_terms = dict(zip(ontology_lst, curr_labels))

In [12]:
# get protein name id
name_path = prj_root / "data" / "protein_name.txt"
with open(name_path, "r") as f:
    prot_lst = f.read().splitlines()# load prot names

In [13]:
# convert protein idx and label idx to protein name and go term respectively
for i, n in enumerate(ontology_lst):
    prot_labels[i] = {
        prot_lst[k]: [namespace_terms[n][x] for x in v]
        for k, v in prot_labels[i].items()
    }

In [14]:
# load performance dataframe
perf_path = prj_root / "data" / "prediction" / "hgat_performance_2.pkl"
perf_df = pd.read_pickle(perf_path)

In [15]:
# load predictions
preddata_path =  [prj_root / "data" / "prediction" / f"hgat_predictions_{x}_bi.pt"
                  for x in ["train", "valid", "test"]]

In [16]:
preddata_bi = [th.load(p) for p in preddata_path]

In [17]:
# drop annotations is None
indices = perf_df["annotations"].apply(lambda x: x is None)
perf_subdf = perf_df[~indices].reset_index(drop=True)

In [18]:
perf_subdf

,split,ontology,proteins,fmax,auprc,annotations
0,train,cc,Q6R327,0.476190,0.610485,"[GO:0031932, GO:0140535, GO:0032991, GO:000562..."
1,train,cc,A0A1D8PEJ5,0.461538,0.833333,"[GO:0005575, GO:0016020, GO:0110165]"
2,train,cc,A6NIX2,0.521739,0.620365,"[GO:0036464, GO:0000932, GO:0005622, GO:004322..."
3,train,cc,A0A1D5PZ68,0.666667,0.526786,"[GO:0005576, GO:0110165, GO:0005575, GO:0005615]"
4,train,cc,A0A0A0MTS5,0.352941,0.664716,"[GO:0012505, GO:0071944, GO:0043229, GO:011016..."
...,...,...,...,...,...,...
238470,test,bp,Q9VT33,0.160000,0.120250,"[GO:0007610, GO:0050896, GO:0042048, GO:000763..."
238471,test,bp,Q9VYA0,0.120000,0.110857,"[GO:0048519, GO:0023052, GO:0099177, GO:000726..."
238472,test,bp,Q9W5X1,0.298507,0.327778,"[GO:0023052, GO:0009966, GO:0010647, GO:004858..."
238473,test,bp,Q9Y320,0.153846,0.142901,"[GO:0007417, GO:0007399, GO:0007275, GO:000742..."


In [19]:
# load embedding
ns = ["cc", "mf", "bp"]
ontology_lst = ["cellular_component", "molecular_function", "biological_process"]
dr_root = prj_root / "data" / "derived_feature"

method_emb = []
method_goemb = []
split_type = ["train", "valid", "test"]

# prot_emb = th.load(dr_root / f"{m}_features.pt")
# go_emb = th.load(dr_root / f"{m}_go_emb.pt")
# for x in ontology_lst:
#     method_emb[x].append(prot_emb[x])
#     method_goemb[x].append(go_emb[x])
protemb_lst = [th.load(dr_root / f"hgat_features_{x}.pt")
               for x in split_type]
# go_emb = th.load(dr_root / f"hgat_go_emb.pt")

# merge the prot emb from train and test
memd_lst = [dict(it.chain.from_iterable(x.items() for x in tp))
            for tp in zip(*protemb_lst)]

# merget the prot emb of all ontologies
# firstly calculate the count of each protein
# the final embedding is the average of the embeddings
counter = clt.Counter(it.chain.from_iterable(x.keys() for x in memd_lst))
h_size = next(iter(memd_lst[0].values())).shape[0]
protemb_dict = clt.defaultdict(lambda: th.zeros(h_size, dtype=th.float32))
for embdict in memd_lst:
    for k, v in embdict.items():
        protemb_dict[k] += v
# for k in protemb_dict:
#     protemb_dict[k] /= counter[k]
print(len(protemb_dict))
# goemb_dict = dict(it.chain.from_iterable(x.items() for x in go_emb.values()))
# method_emb.append(protemb_dict)
# method_goemb.append(goemb_dict)

123534


In [20]:
# load esm embeddings
esm_path = prj_root / "data" / "features.npz"
prot_esm = np.load(esm_path)
prot_esm = {x: e for x, e in zip(prot_lst, prot_esm["features"])}

In [21]:
prot_esm = {x: e for x in protemb_dict.keys()
            if (e := prot_esm.get(x)) is not None}

In [22]:
len(prot_esm)

123534

In [23]:
selected_cases = ea.comprehensive_case_selection_scalable(perf_subdf, prot_esm, protemb_dict,
                                                          max_sample_size=50000)

Starting scalable case selection...
Sampling 50000 proteins from 238475 total proteins...
Selecting performance-based cases...
Finding functional groups...
Processing cc ontology...
Found 20 groups for cc
Processing bp ontology...
Found 20 groups for bp
Processing mf ontology...
Found 20 groups for mf
Finding clustering-based groups...
Clustering cc ontology...
Found 20 quality clusters for cc
Clustering bp ontology...
Found 20 quality clusters for bp
Clustering mf ontology...
Found 20 quality clusters for mf
Analyzing embedding changes...
Preparing embedding dimension alignment...
ESM embeddings shape: (10000, 1280)
Model embeddings shape: (10000, 128)
Applying PCA to reduce ESM embeddings from 1280 to 128 dimensions...
PCA explained variance ratio (first 10 components): [0.09255019 0.05309023 0.03905372 0.0310867  0.0257768  0.02074689
 0.01938636 0.01760105 0.01637474 0.01520235]
Total explained variance: 0.7552
Reduced ESM embeddings shape: (10000, 128)
Processing cc ontology...
  

In [24]:
selected_cases.keys()

dict_keys(['performance_based', 'functional_groups', 'clustering_groups', 'embedding_changes'])

In [25]:
# perf_selected = {}

In [26]:
selected_cases["embedding_changes"]["cc"].keys()

dict_keys(['largest_cosine_distance', 'smallest_cosine_distance', 'largest_l2_distance', 'smallest_l2_distance', 'highest_correlation', 'lowest_correlation', 'all_changes'])

In [27]:
tmp_df = []
for i, n in enumerate(ns):
    sub = selected_cases["embedding_changes"][n]["all_changes"]
    s = len(sub)
    sub_df = pd.DataFrame(sub)
    sub_df["ontology"] = n
    tmp_df.append(sub_df)

changed_df = pd.concat(tmp_df)

In [28]:
changed_df = changed_df.rename(columns={"protein":"proteins"})

In [29]:
anno_counter = clt.Counter(it.chain.from_iterable(changed_df["annotations"]))

In [30]:
# removet the root annotation
BIOLOGICAL_PROCESS = 'GO:0008150'
MOLECULAR_FUNCTION = 'GO:0003674'
CELLULAR_COMPONENT = 'GO:0005575'
anno_counter.pop(BIOLOGICAL_PROCESS)
anno_counter.pop(MOLECULAR_FUNCTION)
anno_counter.pop(CELLULAR_COMPONENT)

3423

In [31]:
fng_df = []
for i, n in enumerate(ns):
    tmp_df = []
    for i, ds in enumerate(selected_cases["functional_groups"][n]):
        df = pd.DataFrame(ds)
        df["functional_group"] = f"{n}_{i}"
        tmp_df.append(df)
        
    sub_df = pd.concat(tmp_df)
    sub_df["ontology"] = n
    fng_df.append(sub_df)

func_group_df = pd.concat(fng_df)

In [32]:
clu_df = []
for i, n in enumerate(ns):
    tmp_df = []
    for i, ds in enumerate(selected_cases["clustering_groups"][n]):
        df = pd.DataFrame(ds)
        df["clustering_group"] = f"{n}_{i}"
        tmp_df.append(df)
    
    sub_df = pd.concat(tmp_df)
    sub_df["ontology"] = n
    clu_df.append(sub_df)

clu_group_df = pd.concat(clu_df).reset_index(drop=True)

In [33]:
clu_group_df = clu_group_df.rename(columns={"protein": "proteins"})

In [34]:
clu_group_df

,proteins,fmax,auprc,annotations,clustering_group,ontology
0,G5EES6,0.571429,0.853571,"[GO:0005622, GO:0110165, GO:0005575, GO:0005737]",cc_0,cc
1,Q580Q2,0.315789,0.689394,"[GO:0005575, GO:0005622, GO:0005737, GO:0110165]",cc_0,cc
2,Q9Y4C4,0.363636,0.875000,"[GO:0005622, GO:0005575, GO:0110165, GO:0005737]",cc_0,cc
3,Q8IP72,0.615385,0.843750,"[GO:0005622, GO:0005575, GO:0110165, GO:0005737]",cc_0,cc
4,Q90XS6,0.666667,0.848214,"[GO:0005622, GO:0005575, GO:0110165, GO:0005737]",cc_0,cc
...,...,...,...,...,...,...
16837,A2RUZ6,0.225000,0.219637,"[GO:0048731, GO:0048534, GO:0048513, GO:004863...",bp_19,bp
16838,Q5RGT2,0.285714,0.273865,"[GO:0048872, GO:0048731, GO:0048534, GO:004851...",bp_19,bp
16839,P56261,0.264463,0.179897,"[GO:0070887, GO:0051707, GO:0033993, GO:000237...",bp_19,bp
16840,Q3UG20,0.318841,0.348891,"[GO:0036230, GO:0006139, GO:0006725, GO:003464...",bp_19,bp


In [35]:
col_names = ["proteins", "clustering_group"]

In [36]:
chng_clu_df = changed_df.merge(clu_group_df[col_names], on="proteins", how="left")

In [37]:
chng_clu_df = chng_clu_df[~chng_clu_df["clustering_group"].isna()]

In [38]:
group_counter = clt.Counter(chng_clu_df["clustering_group"])

In [39]:
chng_clu_df[chng_clu_df["clustering_group"] == "bp_13"]

,proteins,cosine_distance,cosine_similarity,l2_distance,correlation,fmax,auprc,annotations,ontology,clustering_group
646,P0A7I0,1.153402,-0.153402,1.518817,-0.059009,1.000000,1.000000,"[GO:0005622, GO:0005575, GO:0005829, GO:011016...",cc,bp_13
647,P0A7I0,1.153402,-0.153402,1.518817,-0.059009,1.000000,1.000000,"[GO:0005622, GO:0005575, GO:0005829, GO:011016...",cc,bp_13
789,Q14978,1.052911,-0.052911,1.451145,-0.057246,0.800000,0.956881,"[GO:0031981, GO:0005634, GO:0001650, GO:007001...",cc,bp_13
1043,P43897,1.113960,-0.113960,1.492622,-0.140958,0.705882,0.804547,"[GO:0031981, GO:0005634, GO:0070013, GO:004322...",cc,bp_13
1545,A0A0B4K6W9,1.112348,-0.112348,1.491541,-0.089853,0.533333,0.674387,"[GO:0005622, GO:1990904, GO:0032991, GO:000557...",cc,bp_13
2230,M9PCD9,1.030803,-0.030803,1.435829,-0.062068,0.642857,0.744311,"[GO:0031674, GO:0099081, GO:0030018, GO:000562...",cc,bp_13
2325,Q03157,1.007326,-0.007326,1.419384,-0.032433,0.222222,0.924796,"[GO:0043227, GO:0005794, GO:0012505, GO:000562...",cc,bp_13
2415,Q9DBZ5,0.905556,0.094444,1.345776,0.013574,0.526316,0.854543,"[GO:0032991, GO:0005622, GO:0005575, GO:000585...",cc,bp_13
2719,P0CT31,0.919319,0.080681,1.355964,0.130807,0.250000,0.578189,"[GO:0120025, GO:0005938, GO:0099081, GO:009859...",cc,bp_13
2793,Q60876,0.882156,0.117844,1.328274,0.118901,0.545455,0.832589,"[GO:0005622, GO:0005575, GO:0110165, GO:0005737]",cc,bp_13


In [40]:
# load structure plddt data
plddt_path = prj_root / "data" / "protein_plddt_avgRes.csv"
plddt_df = pd.read_csv(plddt_path)

In [41]:
plddt_df = plddt_df.rename(columns={"Protein": "proteins",
                            "Avg_pLDDT": "pLDDT"})

In [42]:
chng_clu_df = chng_clu_df.merge(plddt_df, on="proteins", how="left")

In [105]:
# ind1 = ~chng_clu_df["pLDDT"].isna()
perf_protset = set(perf_subdf["proteins"])
ind1 = chng_clu_df["proteins"].apply(lambda x: x in perf_protset)
ind2 = chng_clu_df["clustering_group"] == "bp_8"
ind3 = chng_clu_df["ontology"] == "bp"
chng_clu_df[ind1 & ind2 & ind3]

,proteins,cosine_distance,cosine_similarity,l2_distance,correlation,fmax,auprc,annotations,ontology,clustering_group,pLDDT
1246,Q0WNR2,0.854238,0.145762,1.307087,0.130832,0.989691,0.989583,"[GO:0006139, GO:0051171, GO:0006725, GO:003464...",bp,bp_8,70.43
1280,Q14119,1.164597,-0.164597,1.526170,-0.033833,0.769231,0.812025,"[GO:0034641, GO:0045595, GO:0006355, GO:000988...",bp,bp_8,58.63
1284,Q6AV35,1.009579,-0.009579,1.420971,0.006087,0.738462,0.979167,"[GO:0006139, GO:0051171, GO:0006725, GO:003464...",bp,bp_8,53.06
1299,P39375,0.948466,0.051534,1.377291,-0.004155,0.482143,0.841792,"[GO:0033554, GO:0006139, GO:0034641, GO:000989...",bp,bp_8,85.34
1310,Q9Y821,1.030929,-0.030929,1.435917,-0.103664,0.622642,0.813786,"[GO:0006139, GO:0034641, GO:0009893, GO:000989...",bp,bp_8,72.97
1317,O00463,1.032046,-0.032046,1.436695,-0.065855,0.666667,0.682941,"[GO:0034641, GO:0006355, GO:0060255, GO:001921...",bp,bp_8,87.05
1345,Q93ZL5,1.123442,-0.123442,1.498961,0.057025,0.941176,0.997685,"[GO:0006139, GO:0051171, GO:0006725, GO:003464...",bp,bp_8,54.70
1349,Q9SGS2,0.979044,0.020956,1.399317,0.200880,0.727273,0.832894,"[GO:0006139, GO:0034641, GO:0006355, GO:006025...",bp,bp_8,73.91
1367,P23890,1.007134,-0.007134,1.419249,0.040782,0.648148,0.722754,"[GO:0006139, GO:0034641, GO:0009893, GO:000989...",bp,bp_8,87.15
1378,Q91WJ8,1.079494,-0.079494,1.469349,-0.143879,0.535714,0.664770,"[GO:0006139, GO:0034641, GO:0006366, GO:000635...",bp,bp_8,64.49


In [53]:
def get_predicted_annotations(pred_data,
                              curr_labels, 
                              by, 
                              name):
    assert isinstance(by, int)
    for pred in pred_data:
        pred_result = None
        assert by < len(pred)
        p = pred[by]
        assert p is not None
        if (tmp := p.get(name)) is not None:
            pred_result = tmp.detach().cpu().numpy()
            break
    targs, preds = pred_result
    assert by < len(curr_labels)
    label_lst = curr_labels[by]
    tp = preds * targs 
    indices = np.where(tp)[0]
    return [label_lst[i] for i in indices]

In [ ]:
name = "Q93ZL5"
# annot1 = get_predicted_annotations(preddata_bi, curr_labels, 2, name)
annot1 = get_predicted_annotations(preddata_bi, curr_labels, 1, name)

In [ ]:
name = "Q9QZQ0"
# annot2 = get_predicted_annotations(preddata_bi, curr_labels, 2, name)
annot2 = get_predicted_annotations(preddata_bi, curr_labels, 1, name)

In [127]:
annot1 = sorted(annot1, key=lambda p:p[1])

In [128]:
annot2 = sorted(annot2, key=lambda x: x[1])

In [129]:
annot_set1 = set(annot1)
annot_set2 = set(annot2)

In [130]:
inter_terms = annot_set1.intersection(annot_set2)

In [133]:
term_count_ic[term_count_ic["gos"].isin(annot2)]

,gos,counts,ic,position,maxdep,lgst
3428,GO:0003674,64279,0.000000,Shallow,17,1
3784,GO:0005488,43653,0.558267,Shallow,17,2
5419,GO:0005515,32354,0.990403,Shallow,14,3
5850,GO:0046983,1947,5.045023,Medium,7,4


In [131]:
term_count_ic[term_count_ic["gos"].isin(inter_terms)]

,gos,counts,ic,position,maxdep,lgst
3428,GO:0003674,64279,0.000000,Shallow,17,1
3784,GO:0005488,43653,0.558267,Shallow,17,2
5419,GO:0005515,32354,0.990403,Shallow,14,3


In [115]:
obo_path = prj_root / "data" / "data-netgo" / "go.obo"
godag = get_godag(obo_path, optional_attrs="relationship")

  EXISTS: /data0/shaojiangyi/pprogo-flg-1/data/data-netgo/go.obo
/data0/shaojiangyi/pprogo-flg-1/data/data-netgo/go.obo: fmt(1.2) rel(2021-01-01) 47,285 Terms; optional_attrs(relationship)


In [122]:
# go_ids = ["GO:0006351", 
#           "GO:0006355",
#           # "GO:1903506",
#         #   "GO:2001141",
#         "GO:0097659",
#           ]
go_ids = list(inter_terms)
gosubdag = GoSubDag(go_ids, godag, relationships=False, prt=False)

In [123]:
saving_dir = prj_root / "notebooks" /"case_analysis" / "go_dag"
saving_path = saving_dir / f"subdag_{'_'.join(go_ids[:5])}.png"
GoSubDagPlot(gosubdag).plt_dag(saving_path)

   48 usr  48 GOs  WROTE: /data0/shaojiangyi/pprogo-flg-1/notebooks/case_analysis/go_dag/subdag_GO:0050789_GO:0008152_GO:0031323_GO:0009987_GO:2001141.png


In [ ]:
gosubdag.

<function goatools.gosubdag.gosubdag.GoSubDag.get_fncsortnt.<locals>.<lambda>(ntgo)>